[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarcusBae/EYE-D/blob/feat/phase1/edge/notebooks/resize_videos.ipynb)

# 영상 해상도 축소 (1920x1080 → 960x540)

Re-ID 파이프라인 처리 속도 향상을 위해 원본 영상을 절반 크기로 변환합니다.

| 항목 | 내용 |
|---|---|
| 입력 | `data/*.avi` (1920x1080, Drive) |
| 인코딩 | `/content/data_960/` (로컬 SSD, 빠름) |
| 출력 | `data_960/*.avi` (Drive 복사) |
| 코덱 | libx264, CRF 18 (고품질) |
| 예상 효과 | 파일 크기 ~75% 감소, 처리 속도 ~3배 향상 |

> **로컬 실행**: 셀 1(Drive 마운트)을 건너뛰고 셀 3의 `DRIVE_ROOT` 경로를 로컬 경로로 변경하세요.

### 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

### 2. 최신 코드 가져오기

In [ ]:
import subprocess, os, shutil

REPO_URL    = 'https://github.com/MarcusBae/EYE-D.git'
REPO_BRANCH = 'feat/phase1'
REPO_LOCAL  = '/content/drive/MyDrive/projects/EYE-D/EYE-D'
TMP_CLONE   = '/content/_ey_d_clone'

print('최신 코드 다운로드 중...')
if os.path.exists(TMP_CLONE):
    shutil.rmtree(TMP_CLONE)

r = subprocess.run(
    ['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, TMP_CLONE],
    text=True, capture_output=True
)
print(r.stdout or r.stderr)
if r.returncode != 0:
    raise RuntimeError(f'git clone 오류: {r.stderr}')

EXCLUDE = {'.git', 'data', 'data_960', 'results', 'tracks', 'dataset'}
os.makedirs(REPO_LOCAL, exist_ok=True)
for item in os.listdir(TMP_CLONE):
    if item in EXCLUDE:
        continue
    src = os.path.join(TMP_CLONE, item)
    dst = os.path.join(REPO_LOCAL, item)
    if os.path.isdir(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)

shutil.rmtree(TMP_CLONE)
print('완료')

### 3. 경로 및 파라미터 설정

In [ ]:
import subprocess, os, pathlib, time, shutil

# ── 경로 설정 ─────────────────────────────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/projects/EYE-D/EYE-D'  # ← 필요 시 수정

DATA_DIR    = pathlib.Path(DRIVE_ROOT) / 'data'      # 원본 영상 (Drive)
LOCAL_OUT   = pathlib.Path('/content/data_960')       # 인코딩 임시 폴더 (SSD)
DRIVE_OUT   = pathlib.Path(DRIVE_ROOT) / 'data_960'  # 최종 저장 (Drive)

WIDTH, HEIGHT = 960, 540
CRF           = 18   # 낮을수록 고품질 (18 = 시각적 무손실 수준)

LOCAL_OUT.mkdir(exist_ok=True)
DRIVE_OUT.mkdir(exist_ok=True)

# FFmpeg 설치 확인
if subprocess.run(['ffmpeg', '-version'], capture_output=True).returncode != 0:
    raise RuntimeError('FFmpeg 없음: sudo apt-get install -y ffmpeg')

videos = sorted(DATA_DIR.glob('*.avi'))
print(f'입력 폴더 : {DATA_DIR}')
print(f'인코딩 위치: {LOCAL_OUT}  (SSD)')
print(f'최종 저장  : {DRIVE_OUT}  (Drive)')
print(f'목표 해상도: {WIDTH}x{HEIGHT}  CRF={CRF}')
print(f'\n처리 대상: {len(videos)}개')
for v in videos:
    print(f'  {v.name}  ({v.stat().st_size / 1024**2:.0f} MB)')

### 4. 원본 영상 정보 확인

In [ ]:
import cv2

print('── 원본 영상 정보 ──────────────────────────────────')
for v in videos:
    cap = cv2.VideoCapture(str(v))
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    mb  = v.stat().st_size / 1024**2
    print(f'  {v.name}: {w}x{h}  {fps:.1f}fps  {n:,}프레임  {mb:.0f}MB')

### 5. 변환 실행

Drive 원본 → `/content/` SSD 인코딩 (Drive FUSE 쓰기 병목 방지)

In [ ]:
total_start = time.time()
results = []

for v in videos:
    local_dst = LOCAL_OUT / v.name
    drive_dst = DRIVE_OUT / v.name

    # Drive에 이미 있으면 건너뜀
    if drive_dst.exists():
        print(f'⏭  건너뜀 (Drive에 이미 존재): {v.name}')
        results.append((v.name, v.stat().st_size, drive_dst.stat().st_size, 0))
        continue

    print(f'▶  인코딩 중: {v.name} ...', end='', flush=True)
    t0 = time.time()

    proc = subprocess.run(
        [
            'ffmpeg', '-y',
            '-i',  str(v),
            '-vf', f'scale={WIDTH}:{HEIGHT}',
            '-c:v', 'libx264',
            '-crf', str(CRF),
            '-preset', 'fast',
            '-c:a', 'copy',
            str(local_dst),
        ],
        capture_output=True, text=True
    )

    encode_time = time.time() - t0

    if proc.returncode != 0:
        print(f'  ✗ 실패')
        print(proc.stderr[-500:])
        continue

    orig_mb  = v.stat().st_size     / 1024**2
    new_mb   = local_dst.stat().st_size / 1024**2
    print(f'  인코딩 완료  {orig_mb:.0f}MB → {new_mb:.0f}MB ({new_mb/orig_mb*100:.0f}%)  {encode_time:.0f}초')

    # SSD → Drive 복사
    print(f'   Drive로 복사 중...', end='', flush=True)
    t1 = time.time()
    shutil.copy2(local_dst, drive_dst)
    local_dst.unlink()  # SSD 임시 파일 삭제
    copy_time = time.time() - t1
    print(f'  완료  ({copy_time:.0f}초)')

    results.append((v.name, v.stat().st_size, drive_dst.stat().st_size, encode_time + copy_time))

print(f'\n총 소요: {(time.time()-total_start)/60:.1f}분')

### 6. 결과 요약

In [ ]:
print('── 변환 결과 요약 ──────────────────────────────────')
total_orig = total_new = 0
for name, orig, new, t in results:
    total_orig += orig
    total_new  += new
    tag = f'{t:.0f}초' if t > 0 else '건너뜀'
    print(f'  {name}: {orig/1024**2:.0f}MB → {new/1024**2:.0f}MB  ({new/orig*100:.0f}%)  {tag}')

if total_orig > 0:
    print(f'\n합계 : {total_orig/1024**2:.0f}MB → {total_new/1024**2:.0f}MB  ({total_new/total_orig*100:.0f}%)')
    print(f'절감 : {(total_orig-total_new)/1024**2:.0f}MB')

print(f'\n출력 폴더: {DRIVE_OUT}')
print('\n다음 단계: run_pipeline_*.ipynb 에서 DATA_DIR 경로를 data_960/ 으로 변경하세요.')